In [1]:
import warnings
from numba.core.errors import NumbaWarning

warnings.simplefilter('ignore', category=NumbaWarning)

import IPython.display as ipd
import torch
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=1023)
           2	RESUME(arg=0, lineno=1023)
           4	LOAD_FAST(arg=0, lineno=1026)
           6	LOAD_CONST(arg=1, lineno=1026)
           8	BINARY_SUBSCR(arg=None, lineno=1026)
          12	LOAD_FAST(arg=0, lineno=1026)
          14	LOAD_CONST(arg=2, lineno=1026)
          16	BINARY_SUBSCR(arg=None, lineno=1026)
          20	COMPARE_OP(arg=68, lineno=1026)
          24	LOAD_FAST(arg=0, lineno=1026)
          26	LOAD_CONST(arg=1, lineno=1026)
          28	BINARY_SUBSCR(arg=None, lineno=1026)
          32	LOAD_FAST(arg=0, lineno=1026)
          34	LOAD_CONST(arg=3, lineno=1026)
          36	BINARY_SUBSCR(arg=None, lineno=1026)
          40	COMPARE_OP(arg=92, lineno=1026)
          44	BINARY_OP(arg=1, lineno=1026)
          48	RETURN_VALUE(arg=None, lineno=1026)
DEBUG:numba.core.byteflow:pending: deque([State(pc_initial=0 nstack_initial=0)])
DEBUG:numba.core.byteflow:stack: []
DEBUG:numba.core.byteflow:state.

/home/yash7/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:numexpr.utils:NumExpr defaulting to 16 threads.
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.piping.pipe(['renderer', 'formatter', 'neato_no_op', 'quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.rendering.render(['renderer', 'formatter', 'neato_no_op', 'quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.unflattening.unflatten(['stagger', 'fanout', 'chain', 'encoding'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.viewing.view(['quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.quote(['is_html_string', 'is_valid_id', 'dot_keywords', 'endswith_odd_number_of_backslashes', 'escape_unescaped_quotes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.a_list(['kwargs', 'attributes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.attr_list(['kwargs', 'attributes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.dot.Dot.clear(

[NeMo W 2025-04-24 04:49:41 nemo_logging:405] /home/yash7/.local/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


DEBUG:torio._extension.utils:Loading FFmpeg6
DEBUG:torio._extension.utils:Failed to load FFmpeg6 extension.
Traceback (most recent call last):
  File "/home/yash7/.local/lib/python3.12/site-packages/torio/_extension/utils.py", line 116, in _find_ffmpeg_extension
    ext = _find_versionsed_ffmpeg_extension(ffmpeg_ver)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yash7/.local/lib/python3.12/site-packages/torio/_extension/utils.py", line 108, in _find_versionsed_ffmpeg_extension
    _load_lib(lib)
  File "/home/yash7/.local/lib/python3.12/site-packages/torio/_extension/utils.py", line 94, in _load_lib
    torch.ops.load_library(path)
  File "/home/yash7/.local/lib/python3.12/site-packages/torch/_ops.py", line 1357, in load_library
    ctypes.CDLL(path)
  File "/home/yash7/miniconda3/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libavutil.so.58: cannot ope

In [ ]:
from speech_reconstruction import get_model

model = get_model().cuda()
model.eval()
_ = utils.load_checkpoint('G_181000.pth', model, None)

INFO:root:Loaded checkpoint 'G_181000.pth' (iteration 1)


In [ ]:
from speech_reconstruction import CustomDataset, DataCollate

dataset = CustomDataset()
collater = DataCollate()
dataLoader = DataLoader(dataset, collate_fn=collater, num_workers=1, shuffle=False, batch_size=1, pin_memory=True, drop_last=False)
data_list = list(dataLoader)

In [ ]:
with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    print(spec.shape, spec_lengths)
    sid_tgt1 = torch.LongTensor([38]).cuda()
    sid_tgt2 = torch.LongTensor([53]).cuda()
    sid_tgt3 = torch.LongTensor([54]).cuda()
    audio1 = model.infer(y, y_lengths, sid_tgt=sid_tgt1)[0,0].data.cpu().float().numpy()
    audio2 = model.infer(y, y_lengths, sid_tgt=sid_tgt2)[0,0].data.cpu().float().numpy()
    audio3 = model.infer(y, y_lengths, sid_tgt=sid_tgt3)[0,0].data.cpu().float().numpy()
    audio4 = model.infer(y, y_lengths, sid_tgt=sid_src)[0,0].data.cpu().float().numpy()
    print(y.shape, audio1.shape, audio2.shape, audio3.shape)    
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=24_000, normalize=True))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=24_000, normalize=True))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=24_000, normalize=True))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=24_000, normalize=True))
print("Converted SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio4, rate=24_000, normalize=True))